# Week 17, Day 4: Analysing loops

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w017d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
n = 20
count = 0
for i in range(n):
    for j in range(i, n):
        count += 1
print(count)

In [ ]:
def count_runs(n):
    a = sum(1 for i in range(0, n, 3))

    b = 0
    i = 1
    while i < n:
        i *= 2
        b += 1

    c = 0
    i = n
    while i > 0:
        i //= 10
        c += 1
    return a, b, c

for n in (1000, 1_000_000):
    print(n, count_runs(n))

*Interactive figure on the website: Each row doubles n. Linear loops double the count, quadratic ones multiply it by 4, logarithmic ones add one..*

In [ ]:
def is_prime(k):
    if k < 2:
        return False
    d = 2
    while d * d <= k:            # O(√k)
        if k % d == 0:
            return False
        d += 1
    return True

def primes_up_to(n):
    return [k for k in range(n) if is_prime(k)]    # n calls, each up to O(√n)

print(primes_up_to(50))
print(len(primes_up_to(100_000)))

In [ ]:
def merge(a, b):
    """Merge two sorted lists into one sorted list."""
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    result += a[i:] + b[j:]
    return result

print(merge([1, 4, 9, 12], [2, 3, 10]))

In [ ]:
def longest_run(items):
    """Length of the longest stretch of equal neighbours."""
    best = 0
    i = 0
    while i < len(items):
        j = i
        while j < len(items) and items[j] == items[i]:
            j += 1
        best = max(best, j - i)
        i = j                       # jump past the whole run
    return best

print(longest_run([1, 1, 2, 2, 2, 3, 1, 1]))

> **Key idea.** Loops in sequence add; loops inside loops multiply; a doubling or halving counter gives log n; and for a while loop, count the total distance its counters can travel.

> **Common mistake.** Two loops in a function don't make it O(n²) unless one is **inside** the other. And one loop doesn't make it O(n) if its body hides an O(n) operation. Always look at what's inside.

### Exercise: How many times?

For each loop, give the exact number of times `body()` runs when n = 100.

In [ ]:
# a: for i in range(0, n, 5): body()
# b: i = 1; while i < n: body(); i *= 3
# c: for i in range(n): for j in range(i, i + 4): body()
# d: for i in range(n): for j in range(i + 1, n): body()
# e: i = n; while i > 1: body(); i //= 2
a = ___
b = ___
c = ___
d = ___
e = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 20, \"a\"\n    assert b == 5, \"b: 1, 3, 9, 27, 81\"\n    assert c == 400, \"c\"\n    assert d == 4950, \"d\"\n    assert e == 6, \"e: 100, 50, 25, 12, 6, 3\"")

### Exercise: Class each loop

Give the Big-O of each function in n as a string: `"O(1)"`, `"O(log n)"`, `"O(sqrt n)"`, `"O(n)"`, `"O(n log n)"`, `"O(n^2)"` or `"O(n^3)"`.

In [ ]:
def f1(n):
    total = 0
    for i in range(n):
        for j in range(100):
            total += i * j
    return total

def f2(n):
    total = 0
    i = n
    while i > 0:
        for j in range(n):
            total += 1
        i //= 2
    return total

def f3(n):
    total = 0
    for i in range(n):
        total += i
    for i in range(n):
        for j in range(n):
            total += 1
    return total

def f4(n):
    d = 2
    while d * d <= n and n % d:
        d += 1
    return d

def f5(n):
    total = 0
    for i in range(n):
        for j in range(i):
            for k in range(j):
                total += 1
    return total

a1 = ___
a2 = ___
a3 = ___
a4 = ___
a5 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a1 == \"O(n)\", \"f1\"\n    assert a2 == \"O(n log n)\", \"f2\"\n    assert a3 == \"O(n^2)\", \"f3\"\n    assert a4 == \"O(sqrt n)\", \"f4\"\n    assert a5 == \"O(n^3)\", \"f5\"")

### Exercise: Measure the ratio

Write `body_runs(n)` that returns how many times the innermost line of this loop runs, by adding a counter:

```
for i in range(1, n + 1):
    j = i
    while j <= n:
        (innermost line)
        j += i
```

Then set `ratio` to `body_runs(20000) / body_runs(10000)` rounded to 2 decimal places, and `cls` to its class as a string. (The inner loop runs about n/i times; n/1 + n/2 + n/3 + ... is about n times the natural logarithm of n.)

In [ ]:
ratio = ___
cls = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert body_runs(1) == 1 and body_runs(4) == 8 and body_runs(10) == 27\n    assert ratio == round(body_runs(20000) / body_runs(10000), 2)\n    assert cls == \"O(n log n)\"")

### Exercise: Runs in O(n)

Write `runs(text)` that returns a list of `(character, length)` pairs for each run of equal characters in a string, in order: `"aaabccdddd"` gives `[("a", 3), ("b", 1), ("c", 2), ("d", 4)]`. Use the "jump past the whole run" idea so it's O(n): a while loop with an inner while loop, where the outer position jumps to where the inner one stopped.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert runs(\"aaabccdddd\") == [(\"a\", 3), (\"b\", 1), (\"c\", 2), (\"d\", 4)]\n    assert runs(\"\") == [] and runs(\"z\") == [(\"z\", 1)]\n    big = \"ab\" * 100_000 + \"c\" * 300_000\n    r = runs(big)\n    assert len(r) == 200_001 and r[-1] == (\"c\", 300_000)")

### Exercise: Merging, counted

Write `merge_counting(a, b)` that merges two sorted lists (as in the lesson) and returns `(merged, passes)`, where `passes` is how many times the main while loop ran. Then set `max_passes` to a formula, as a function `max_passes(n, m)`, for the largest number of passes possible with lists of lengths n and m (both at least 1).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert merge_counting([1, 4, 9], [2, 3, 10]) == ([1, 2, 3, 4, 9, 10], 5)\n    assert merge_counting([1, 2, 3], [7, 8]) == ([1, 2, 3, 7, 8], 3)\n    assert merge_counting([], [5]) == ([5], 0)\n    for n, m in ((3, 3), (1, 5), (10, 4)):\n        a = list(range(0, 2 * n, 2))\n        b = list(range(1, 2 * m, 2))\n        assert merge_counting(a, b)[1] <= max_passes(n, m)\n    assert max_passes(3, 3) == 5 and max_passes(1, 5) == 5")

### Bug hunt: Prefix sums

`running_totals(items)` returns the total of the first 1, 2, 3, ... items. It's correct but O(n²), because `sum(items[:i + 1])` adds up the slice from scratch each time. Make it O(n), keeping one running total.

In [ ]:
def running_totals(items):
    return [sum(items[:i + 1]) for i in range(len(items))]

print(running_totals([3, 1, 4, 1, 5]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"same, O(n)\"\n    expect_output(\"[3, 4, 8, 9, 14]\")\n    assert running_totals([]) == []\n    assert running_totals([1] * 200_000)[-1] == 200_000\n    assert \"sum(\" not in source")

### Bug hunt: Halving that goes on and on

This should count how many times `n` can be halved before reaching 0 (it's O(log n)), but it prints counts of over a thousand. Fix it so `halvings(0)` is 0, `halvings(1)` is 1 and `halvings(8)` is 4.

In [ ]:
def halvings(n):
    count = 0
    while n != 0:
        n = n / 2
        count += 1
    return count

print(halvings(0), halvings(1), halvings(8))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"0 1 4\"\n    expect_output(\"0 1 4\")")

### Puzzle: An O(n log n) table

Arrange the lines so the function prints a times table of the powers of two below `n` for each `i` below `n`: an O(n log n) pair of loops. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # Big-O of: for x in items: if x in items[:5]: ...  (as a string)
b = ___   # Big-O of: for x in a: for y in b: ...  with len(a) = n, len(b) = m
c = ___   # body runs of: for i in range(10): for j in range(i): body()
d = ___   # True or False: a while loop with two positions that each only move towards each other is O(n)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"O(n)\" and b.replace(\" \", \"\") in (\"O(n*m)\", \"O(nm)\", \"O(m*n)\", \"O(mn)\") and c == 45 and d is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: longest stretch under a limit

Given a list of positive whole numbers and a limit, find the length of the longest stretch of neighbouring items whose total is at most the limit. Write three versions:

- `longest_cubic(items, limit)`: try every start and every end, adding the stretch up with `sum` each time: O(n³).
- `longest_quadratic(items, limit)`: for every start, extend the end one at a time, keeping a running total, and stop extending once the total is over the limit: O(n²).
- `longest_linear(items, limit)`: a sliding window with two positions, `start` and `end`, both only moving forward. Add `items[end]` to the total; while the total is over the limit, take away `items[start]` and move `start` up. The window's length is a candidate each time: O(n), since each position moves at most n times.

It must handle 100,000 items quickly with the linear version.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"all agree\"\n    assert longest_linear([2, 1, 3, 4, 1, 1, 1], 5) == 3\n    assert longest_linear([10, 20], 5) == 0 and longest_linear([], 5) == 0\n    rng = random.Random(5)\n    for _ in range(40):\n        data = [rng.randint(1, 9) for _ in range(rng.randint(0, 30))]\n        lim = rng.randint(1, 30)\n        a, b, c = longest_cubic(data, lim), longest_quadratic(data, lim), longest_linear(data, lim)\n        assert a == b == c, (data, lim, a, b, c)\ndef test_big():\n    \"big\"\n    rng = random.Random(9)\n    data = [rng.randint(1, 100) for _ in range(100_000)]\n    assert longest_linear(data, 5000) == 122")

## Warm-up

When `i` is 0 the inner loop runs 20 times, when `i` is 1 it runs 19 times, down to 1 time when `i` is 19. That's 20 + 19 + ... + 1 = 210, which is n(n + 1)/2. Still O(n²): about half of n², and Big-O drops the half.

## Single loops

The number of times a loop's body runs is what matters, and the cost of each run.

**Counting up in steps:** `for i in range(0, n, 3)` runs about n/3 times. That's still O(n): any fixed step size is a constant factor.

**Doubling or halving:** a counter that doubles each time reaches n after about log₂ n steps, so the loop is O(log n). The same for halving down to 1, or for dividing by 10 (log₁₀ n, still O(log n)).

Multiplying n by 1,000 multiplied the first count by 1,000 but only **added** about 10 to the second and 3 to the third.

**Stopping early:** a loop with `break` or `return` inside can finish early, but in the worst case it doesn't. Analyse the worst case unless asked otherwise (Day 5 is about the difference).

## Nested loops

**Independent ranges multiply:** an outer loop of n and an inner loop of m is O(n × m). If both are n, that's O(n²). If the inner range is a **constant** (`range(10)`), the inner loop is O(1) and the whole thing is O(n).

**Inner range depends on the outer:** `for j in range(i)` runs 0, 1, 2, ..., n − 1 times: n(n − 1)/2 in total, still O(n²). The triangle shape is half the square, and halves don't change the class.

**Inner loop doubles:** an outer loop of n with an inner doubling loop of log n each time is O(n log n).

Pick a loop shape, look at the counts as n doubles, and decide the class before revealing it:

The `sqrt` loop is a class we haven't named: `while i * i <= n` stops once i reaches √n, so it's **O(√n)**, between O(log n) and O(n). Checking whether a number is prime by trying divisors up to its square root is O(√n).

## Sequences and calls

**Loops one after another add:** an O(n) loop followed by an O(n²) loop is O(n + n²) = O(n²). The biggest part wins.

**A call inside a loop multiplies:** the body's cost includes whatever it calls. A loop of n that calls an O(n) function each time is O(n²), even though there's only one `for` in sight:

`primes_up_to(n)` is O(n√n): n calls, each at most about √n steps.

The hidden costs from Day 2 count too. A comprehension with `x in some_list` inside it, `items.index(x)` inside a loop, or `sum(items[:i])` for each i are all quiet O(n²)s.

## While loops

For a `while` loop, ask: **what moves closer to the end on each pass, and by how much?**

It's not obvious how many times this runs, but each pass moves **either** `i` **or** `j` forward by one, and neither can move more than len(a) or len(b) times in total. So there are at most len(a) + len(b) passes: **O(n + m)**. The same reasoning shows two-position loops (lo and hi moving towards each other) are O(n): every pass shrinks the gap between them by at least one.

This "count the total moves" idea also explains why some nested loops are faster than they look:

A loop inside a loop, but it's O(n): `i` jumps straight to where `j` stopped, so `j` moves forward across each item exactly once in total.

## A checklist

To analyse a function:

1. Find every loop (including hidden ones: comprehensions, `in`, `sum`, `sorted`, slicing, `count`, `index`, `insert(0, ...)`).
2. For each loop, ask how many times it runs: n? n/k? log n? Depends on an outer loop? Total distance travelled?
3. Multiply by the cost of the body, including any calls.
4. Add up the parts in sequence, and keep the biggest.
5. Check by counting: add a counter, run for n and 2n, and see if the ratio matches.

## Practice

## Quiz

## Daily challenge

## Before you go

Find the most complicated loop you've written so far in this course and analyse it with the checklist, writing the reasoning as a comment above it. Then add a counter and check your answer with the doubling test. Tomorrow: best and worst cases, and space, the memory an algorithm needs.